# External Lab RefMet Final Reproducible Evaluation

This notebook is the canonical lab-name evaluation run for ChemSynonymizer. It consolidates the deterministic CID-mapping work and the corrected closed-set `metabolite_name -> refmet_name` evaluation into one reproducible Colab workflow.

It supersedes the earlier exploratory `external_lab_refmet_*` notebooks for reporting purposes. All experiment outputs are written under a single run directory:

`results/external_lab_refmet_final_reproducible/`

The notebook performs:

1. lab-name cleaning and pair deduplication,
2. deterministic normalized-name CID mapping against the PubMed/PubChem synonym reference,
3. closed-set RefMet target construction,
4. SapBERT candidate generation,
5. embedding-only and hybrid-reranked evaluation,
6. confidence/error analysis,
7. final figures, source table, artifact manifest, and a short research-update report.


## Install Dependencies


In [ ]:
import subprocess
import sys

base_packages = [
    'pandas==2.2.2',
    'numpy>=1.26.4,<2.1',
    'pyarrow>=15.0',
    'duckdb>=1.0',
    'tqdm>=4.66',
    'torch>=2.1',
    'transformers>=4.40',
    'accelerate>=0.28',
    'sentencepiece>=0.2',
    'matplotlib>=3.8',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *base_packages])

try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-gpu-cu12'])
except subprocess.CalledProcessError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu>=1.8'])

print('Installed notebook dependencies.')


## Mount Drive and Locate Repo


In [ ]:
from pathlib import Path
import gc
import json
import os
import re
import shutil
import sys
import time
import unicodedata
from difflib import SequenceMatcher

import duckdb
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)


def find_repo_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content/drive/MyDrive/ChemSynonymizer/chem_synonymizer'),
        Path('/content/drive/MyDrive/chem_synonymizer'),
        Path('/content/chem_synonymizer'),
    ]
    for candidate in candidates:
        if (candidate / 'src' / 'chem_synonymizer').exists():
            return candidate.resolve()
    raise FileNotFoundError('Could not find repo root containing src/chem_synonymizer.')


REPO_DIR = find_repo_dir()
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo:', REPO_DIR)


## Configuration


In [ ]:
from chem_synonymizer.embed import load_hf_model, pool_outputs
from chem_synonymizer.utils import save_json

LAB_CSV_PATH = REPO_DIR / 'data' / 'allmetnames_062626.csv'
REFERENCE_CSV_PATH = REPO_DIR / 'data' / 'CID-Synonym-full.csv'
MODEL_DIR = REPO_DIR / 'results' / 'sapbert_contrastive' / 'artifacts' / 'model'

METABOLITE_COL = 'metabolite_name'
REFMET_COL = 'refmet_name'
CID_COL = 'cid'
NAME_COL = 'name'

RUN_NAME = 'external_lab_refmet_final_reproducible'
RESULTS_RUN_DIR = REPO_DIR / 'results' / RUN_NAME
RESULTS_ARTIFACTS_DIR = RESULTS_RUN_DIR / 'artifacts'

# Final notebook writes all reusable artifacts into this one run folder.
SHARED_ARTIFACT_DIRS = []

SCRATCH_ROOT = Path('/mnt/local-scratch') if Path('/mnt/local-scratch').exists() else Path('/content')
WORK_DIR = SCRATCH_ROOT / 'chem_synonymizer' / RUN_NAME
WORK_ARTIFACTS_DIR = RESULTS_ARTIFACTS_DIR

CORPUS_PARQUET_PATH = RESULTS_ARTIFACTS_DIR / 'targeted_reference_lookup.parquet'
QUERY_EMBEDDINGS_PATH = WORK_ARTIFACTS_DIR / 'metabolite_query_embeddings.npy'
REFMET_EMBEDDINGS_PATH = WORK_ARTIFACTS_DIR / 'refmet_label_embeddings.npy'
NEIGHBOR_INDICES_PATH = WORK_ARTIFACTS_DIR / 'closed_refmet_neighbor_indices.npy'
NEIGHBOR_SCORES_PATH = WORK_ARTIFACTS_DIR / 'closed_refmet_neighbor_scores.npy'

SEED = 7
TRAIN_FRAC = 0.60
VAL_FRAC = 0.20
TEST_FRAC = 0.20

MAX_LENGTH = 96
EMBED_DTYPE = 'float16'
POOLING = 'cls'
L2_NORMALIZE = True
QUERY_ENCODE_BATCH_SIZE = 8192
REFMET_ENCODE_BATCH_SIZE = 8192
SEARCH_BATCH_SIZE = 8192
K_RETRIEVE = 256

# Final run: rerank the full top-256 pool.
RERANK_POOL_K = 256
TUNE_VAL_SAMPLE_SIZE = 10000
TUNE_RANDOM_WEIGHT_TRIALS = 100
RERANK_BATCH_SIZE = 750
SAVE_TOP_K = 10

# Deterministic mapping against the 114M-name reference.
RUN_DETERMINISTIC_ROW_CID_MAPPING = True
RERUN_DETERMINISTIC_MAPPING = False
REFERENCE_CHUNK_SIZE = 2_000_000

# RefMet-to-CID mapping for identity-aware scoring.
BUILD_CORPUS_IF_MISSING = True
RUN_COMPACT_REFMET_MAPPING = True
MIN_COMPACT_KEY_LEN = 6
MAX_COMPACT_MATCHES_PER_REFMET = 25

REENCODE_QUERIES = False
REENCODE_REFMET_LABELS = False
RERUN_SEARCH = False
RERUN_TUNING = True
RERUN_RERANK_EVAL = True

for required in [LAB_CSV_PATH, MODEL_DIR]:
    if not required.exists():
        raise FileNotFoundError(required)
if not REFERENCE_CSV_PATH.exists():
    print('Warning: reference CSV is missing. Identity-aware same-CID scoring will be limited.')
for path in [RESULTS_RUN_DIR, RESULTS_ARTIFACTS_DIR, WORK_ARTIFACTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Lab CSV:', LAB_CSV_PATH)
print('Reference CSV:', REFERENCE_CSV_PATH)
print('Model:', MODEL_DIR)
print('Results:', RESULTS_RUN_DIR)
print('Artifact storage:', WORK_ARTIFACTS_DIR)
print('Scratch root:', SCRATCH_ROOT, shutil.disk_usage(SCRATCH_ROOT))


## Text and Artifact Helpers


In [ ]:
GREEK_MAP = {
    'α': 'alpha', 'β': 'beta', 'γ': 'gamma', 'δ': 'delta',
    'ε': 'epsilon', 'κ': 'kappa', 'λ': 'lambda', 'μ': 'mu',
    'ω': 'omega',
}


def normalize_text(value: object) -> str:
    if pd.isna(value):
        return ''
    text = unicodedata.normalize('NFKC', str(value))
    for src, dst in GREEK_MAP.items():
        text = text.replace(src, dst)
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    return text


def normalize_series(values: pd.Series) -> pd.Series:
    return values.fillna('').astype('string').map(normalize_text).astype('string')


def clean_instrument_suffix(text: object) -> str:
    s = normalize_text(text)
    s = re.sub(r'\[[^\]]*m[^\]]*\]', ' ', s)
    s = re.sub(r'[@_]\s*[-+]?\d+(?:\.\d+)?\s*$', ' ', s)
    s = re.sub(r'\s+[+-]\s*\d+(?:\.\d+)?\s*$', ' ', s)
    s = s.replace('*', ' ')
    return re.sub(r'\s+', ' ', s).strip()


def punctuation_spaced(text: object) -> str:
    s = clean_instrument_suffix(text)
    s = re.sub(r'[\(\)\[\]\{\};,/_]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def compact_key(text: object) -> str:
    return re.sub(r'[^a-z0-9]+', '', normalize_text(text))


def make_refmet_variants(text: object) -> list[tuple[str, str]]:
    base = normalize_text(text)
    candidates = [
        ('exact', base),
        ('strip_instrument_suffix', clean_instrument_suffix(base)),
        ('punctuation_spaced', punctuation_spaced(base)),
        ('remove_quotes', re.sub(r'[\'"`]', '', base)),
        ('remove_terminal_count', re.sub(r'(?:\s|_)+(?:\d+)$', '', base).strip()),
    ]
    out = []
    seen = set()
    for method, value in candidates:
        value = normalize_text(value)
        if value and value not in seen:
            out.append((method, value))
            seen.add(value)
    return out


def pipe_split(value: object) -> set[str]:
    if pd.isna(value):
        return set()
    return {part.strip() for part in str(value).split('|') if part.strip()}


def pipe_join(values) -> str:
    out = []
    seen = set()
    for value in values:
        for part in pipe_split(value):
            if part not in seen:
                seen.add(part)
                out.append(part)
    return '|'.join(out)


def cid_overlap(a: object, b: object) -> bool:
    return bool(pipe_split(a) & pipe_split(b))


def char_ngrams(text: str, n: int = 3) -> set[str]:
    s = compact_key(text)
    if not s:
        return set()
    if len(s) <= n:
        return {s}
    return {s[i:i+n] for i in range(len(s) - n + 1)}


def token_set(text: str) -> set[str]:
    return {tok for tok in re.split(r'[^a-z0-9]+', normalize_text(text)) if tok}


def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / max(1, len(a | b))


def sequence_ratio(a: str, b: str) -> float:
    a = normalize_text(a)
    b = normalize_text(b)
    if len(a) > 120 or len(b) > 120:
        return jaccard(char_ngrams(a, 3), char_ngrams(b, 3))
    return SequenceMatcher(None, a, b).ratio()


def lipid_signature(text: str) -> tuple[str, str, str] | None:
    s = normalize_text(text)
    m = re.search(r'\b([a-z]{1,6})\s*[\( ]?\s*(\d{1,2})\s*:\s*(\d{1,2})\b', s)
    if not m:
        return None
    return (m.group(1), m.group(2), m.group(3))


def stereo_signature(text: str) -> tuple[str, ...]:
    s = normalize_text(text)
    toks = re.findall(r'\b(?:cis|trans|alpha|beta)\b|\b[dl]-|\([+-]\)|\b[0-9]+[rs]\b|\b[0-9]+[ez]\b', s)
    return tuple(sorted(toks))


def text_profile(text: str) -> dict:
    return {
        'norm': normalize_text(text),
        'compact': compact_key(text),
        'char3': char_ngrams(text, 3),
        'tokens': token_set(text),
        'lipid': lipid_signature(text),
        'stereo': stereo_signature(text),
    }


def lexical_feature_dict(query_profile: dict, candidate_profile: dict) -> dict:
    q = query_profile
    c = candidate_profile
    return {
        'compact_exact': float(q['compact'] == c['compact'] and q['compact'] != ''),
        'sequence_ratio': float(sequence_ratio(q['norm'], c['norm'])),
        'char3_jaccard': float(jaccard(q['char3'], c['char3'])),
        'token_jaccard': float(jaccard(q['tokens'], c['tokens'])),
        'lipid_signature_match': float(q['lipid'] is not None and q['lipid'] == c['lipid']),
        'lipid_class_match': float(q['lipid'] is not None and c['lipid'] is not None and q['lipid'][0] == c['lipid'][0]),
        'stereo_mismatch': float(bool(q['stereo'] and c['stereo'] and q['stereo'] != c['stereo'])),
    }


def strong_alias_match(a: str, b: str) -> bool:
    if normalize_text(a) == normalize_text(b):
        return True
    ca = compact_key(a)
    cb = compact_key(b)
    if ca and ca == cb and min(len(ca), len(cb)) >= 5:
        return True
    seq = sequence_ratio(a, b)
    tok = jaccard(token_set(a), token_set(b))
    ch = jaccard(char_ngrams(a, 3), char_ngrams(b, 3))
    return bool(seq >= 0.96 or (seq >= 0.90 and tok >= 0.80) or (ch >= 0.92 and tok >= 0.70))


def q(path: Path) -> str:
    return str(path).replace("'", "''")


def load_json_if_exists(path: Path) -> dict | None:
    if not path.exists():
        return None
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


def artifact_candidates(work_path: Path) -> list[Path]:
    return [RESULTS_ARTIFACTS_DIR / work_path.name] + [d / work_path.name for d in SHARED_ARTIFACT_DIRS]


def parts_dir_candidates(work_path: Path) -> list[Path]:
    return [p.with_suffix(p.suffix + '.parts') for p in artifact_candidates(work_path)]


def restore_artifact(work_path: Path) -> bool:
    if work_path.exists():
        return True
    for candidate in artifact_candidates(work_path):
        if candidate.exists():
            work_path.parent.mkdir(parents=True, exist_ok=True)
            print('Restoring artifact:', candidate, '->', work_path)
            shutil.copy2(candidate, work_path)
            for suffix in ['.meta.json', '.timing.json']:
                sidecar = candidate.with_suffix(candidate.suffix + suffix)
                if sidecar.exists():
                    shutil.copy2(sidecar, work_path.with_suffix(work_path.suffix + suffix))
            return True
    for parts_dir in parts_dir_candidates(work_path):
        if parts_dir.exists():
            parts = sorted(parts_dir.glob('part-*'))
            if not parts:
                continue
            work_path.parent.mkdir(parents=True, exist_ok=True)
            print(f'Reconstructing {work_path.name} from {len(parts)} parts in {parts_dir}...')
            with work_path.open('wb') as out:
                for part in tqdm(parts, desc=f'RESTORE {work_path.name}'):
                    with part.open('rb') as src:
                        shutil.copyfileobj(src, out, length=64 * 1024 * 1024)
            for suffix in ['.meta.json', '.timing.json']:
                sidecar = parts_dir / f'{work_path.name}{suffix}'
                if sidecar.exists():
                    shutil.copy2(sidecar, work_path.with_suffix(work_path.suffix + suffix))
            return True
    return False


def save_config() -> None:
    save_json(
        {
            'run_name': RUN_NAME,
            'task': 'final closed-set lab refmet_name prediction from metabolite_name',
            'lab_csv': str(LAB_CSV_PATH),
            'reference_csv': str(REFERENCE_CSV_PATH),
            'model_dir': str(MODEL_DIR),
            'split': {'seed': SEED, 'train_frac': TRAIN_FRAC, 'val_frac': VAL_FRAC, 'test_frac': TEST_FRAC},
            'max_length': MAX_LENGTH,
            'pooling': POOLING,
            'embedding_dtype': EMBED_DTYPE,
            'k_retrieve': K_RETRIEVE,
            'rerank_pool_k': RERANK_POOL_K,
            'deterministic_cid_mapping': {
                'enabled': RUN_DETERMINISTIC_ROW_CID_MAPPING,
                'reference_chunk_size': REFERENCE_CHUNK_SIZE,
                'lab_name_columns': [METABOLITE_COL, REFMET_COL],
            },
            'identity_aware_scoring': {
                'exact_label': True,
                'same_pubchem_cid_when_mapped': True,
                'strong_alias_match': True,
            },
        },
        RESULTS_RUN_DIR / 'config.json',
    )


save_config()


## Deterministic Row-Level CID Mapping

This step scans the large PubMed/PubChem synonym reference only for normalized lab names from `metabolite_name` and `refmet_name`. It creates row-level exact CID labels, ambiguous mappings, unmatched rows, and a targeted lookup reused later for identity-aware RefMet scoring.


In [ ]:
def join_unique_values(values) -> str:
    out = []
    seen = set()
    for value in values:
        if pd.isna(value):
            continue
        text = str(value).strip()
        if not text or text in seen:
            continue
        seen.add(text)
        out.append(text)
    return '|'.join(out)


def count_pipe_values(value: object) -> int:
    return len(pipe_split(value))


def build_lab_long_table(lab: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    available = [col for col in [METABOLITE_COL, REFMET_COL] if col in lab.columns]
    if not available:
        raise ValueError(f'None of the expected lab name columns are available: {[METABOLITE_COL, REFMET_COL]}')
    long = lab[['_lab_row_id'] + available].melt(
        id_vars=['_lab_row_id'],
        value_vars=available,
        var_name='source_column',
        value_name='lab_name',
    )
    long['lab_name'] = long['lab_name'].astype('string').str.strip()
    long['normalized_name'] = normalize_series(long['lab_name'])
    long = long[long['normalized_name'].str.len() > 0].copy()
    long = long.drop_duplicates(['_lab_row_id', 'source_column', 'normalized_name']).reset_index(drop=True)
    summary = {
        'available_lab_name_columns': available,
        'lab_name_values_used': int(len(long)),
        'target_lab_normalized_names': int(long['normalized_name'].nunique()),
    }
    return long, summary


def build_targeted_reference_lookup(target_names: set[str]) -> tuple[pd.DataFrame, dict]:
    lookup_path = RESULTS_ARTIFACTS_DIR / 'targeted_reference_lookup.csv'
    summary_path = RESULTS_RUN_DIR / 'targeted_reference_lookup_summary.json'
    if lookup_path.exists() and summary_path.exists() and not RERUN_DETERMINISTIC_MAPPING:
        lookup = pd.read_csv(lookup_path, dtype='string')
        summary = load_json_if_exists(summary_path)
        print('Using cached targeted reference lookup:', lookup_path)
        print(json.dumps(summary, indent=2))
        return lookup, summary

    if not REFERENCE_CSV_PATH.exists():
        summary = {
            'reference_csv_available': False,
            'target_lab_normalized_names': int(len(target_names)),
            'reference_rows_read': 0,
            'matched_reference_name_cid_pairs': 0,
            'unique_matched_normalized_names': 0,
            'ambiguous_matched_normalized_names': 0,
        }
        lookup = pd.DataFrame(columns=['normalized_name', 'candidate_cids', 'n_candidate_cids'])
        lookup.to_csv(lookup_path, index=False)
        save_json(summary, summary_path)
        print('Reference CSV missing; deterministic CID mapping will be unavailable.')
        return lookup, summary

    frames = []
    total_rows = 0
    reader = pd.read_csv(
        REFERENCE_CSV_PATH,
        usecols=[CID_COL, NAME_COL],
        dtype='string',
        chunksize=int(REFERENCE_CHUNK_SIZE),
        on_bad_lines='skip',
    )
    for chunk in tqdm(reader, desc='Scanning reference for targeted lab names'):
        total_rows += len(chunk)
        chunk = chunk.dropna(subset=[CID_COL, NAME_COL]).copy()
        chunk['normalized_name'] = normalize_series(chunk[NAME_COL])
        chunk = chunk[chunk['normalized_name'].isin(target_names)]
        if chunk.empty:
            continue
        chunk = chunk.rename(columns={CID_COL: 'cid'})[['normalized_name', 'cid']].drop_duplicates()
        frames.append(chunk)

    if frames:
        ref_pairs = pd.concat(frames, ignore_index=True).drop_duplicates(['normalized_name', 'cid'])
        lookup = (
            ref_pairs.groupby('normalized_name', sort=True)['cid']
            .agg(lambda s: '|'.join(pd.unique(s.astype(str))))
            .reset_index(name='candidate_cids')
        )
        lookup['n_candidate_cids'] = lookup['candidate_cids'].map(count_pipe_values).astype('int32')
    else:
        lookup = pd.DataFrame(columns=['normalized_name', 'candidate_cids', 'n_candidate_cids'])
        ref_pairs = pd.DataFrame(columns=['normalized_name', 'cid'])

    lookup.to_csv(lookup_path, index=False)
    summary = {
        'reference_csv_available': True,
        'target_lab_normalized_names': int(len(target_names)),
        'reference_rows_read': int(total_rows),
        'matched_reference_name_cid_pairs': int(len(ref_pairs)),
        'unique_matched_normalized_names': int(len(lookup)),
        'ambiguous_matched_normalized_names': int((lookup['n_candidate_cids'].astype(int) > 1).sum()) if len(lookup) else 0,
        'lookup_path': str(lookup_path),
    }
    save_json(summary, summary_path)
    print(json.dumps(summary, indent=2))
    return lookup, summary


def run_deterministic_row_cid_mapping() -> dict:
    out_dir = RESULTS_RUN_DIR / 'deterministic_cid_mapping'
    out_dir.mkdir(parents=True, exist_ok=True)
    summary_path = out_dir / 'deterministic_mapping_summary.json'
    if summary_path.exists() and not RERUN_DETERMINISTIC_MAPPING:
        summary = load_json_if_exists(summary_path)
        print('Using cached deterministic mapping:', summary_path)
        print(json.dumps(summary, indent=2))
        return summary

    lab = pd.read_csv(LAB_CSV_PATH, dtype='string', engine='python', on_bad_lines='skip')
    lab = lab.reset_index(drop=True).copy()
    lab['_lab_row_id'] = np.arange(len(lab), dtype=np.int64)
    lab_long, lab_long_summary = build_lab_long_table(lab)
    target_names = set(lab_long['normalized_name'].dropna().astype(str).unique())
    reference_lookup, reference_summary = build_targeted_reference_lookup(target_names)

    lab_matches_long = lab_long.merge(reference_lookup, on='normalized_name', how='left')
    lab_matches_long['is_exact_match'] = lab_matches_long['candidate_cids'].notna()
    matched_only = lab_matches_long[lab_matches_long['is_exact_match']].copy()

    if matched_only.empty:
        row_matches = pd.DataFrame({'_lab_row_id': lab['_lab_row_id']})
        row_matches['candidate_cids'] = ''
        row_matches['matched_source_columns'] = ''
        row_matches['matched_lab_names'] = ''
        row_matches['matched_normalized_names'] = ''
        row_matches['n_candidate_cids'] = 0
    else:
        row_matches = (
            matched_only.groupby('_lab_row_id', sort=False)
            .agg(
                candidate_cids=('candidate_cids', pipe_join),
                matched_source_columns=('source_column', join_unique_values),
                matched_lab_names=('lab_name', join_unique_values),
                matched_normalized_names=('normalized_name', join_unique_values),
            )
            .reset_index()
        )
        row_matches['n_candidate_cids'] = row_matches['candidate_cids'].map(count_pipe_values).astype('int32')

    mapped_lab = lab.merge(row_matches, on='_lab_row_id', how='left')
    for col in ['candidate_cids', 'matched_source_columns', 'matched_lab_names', 'matched_normalized_names']:
        mapped_lab[col] = mapped_lab[col].fillna('')
    mapped_lab['n_candidate_cids'] = mapped_lab['n_candidate_cids'].fillna(0).astype('int32')
    mapped_lab['mapping_status'] = np.select(
        [
            mapped_lab['n_candidate_cids'].eq(0),
            mapped_lab['n_candidate_cids'].eq(1),
            mapped_lab['n_candidate_cids'].gt(1),
        ],
        ['unmatched', 'exact_unique_cid', 'exact_ambiguous_cid'],
        default='unmatched',
    )

    mapping_summary = {
        'lab_rows': int(len(mapped_lab)),
        'lab_name_values_used': int(lab_long_summary['lab_name_values_used']),
        'matched_rows': int(mapped_lab['n_candidate_cids'].gt(0).sum()),
        'unique_cid_rows': int(mapped_lab['mapping_status'].eq('exact_unique_cid').sum()),
        'ambiguous_cid_rows': int(mapped_lab['mapping_status'].eq('exact_ambiguous_cid').sum()),
        'unmatched_rows': int(mapped_lab['mapping_status'].eq('unmatched').sum()),
        'unique_assigned_cids': int(mapped_lab.loc[mapped_lab['mapping_status'].eq('exact_unique_cid'), 'candidate_cids'].nunique()),
    }
    mapping_summary['matched_row_pct'] = round(100 * mapping_summary['matched_rows'] / max(mapping_summary['lab_rows'], 1), 3)
    mapping_summary['unique_cid_row_pct'] = round(100 * mapping_summary['unique_cid_rows'] / max(mapping_summary['lab_rows'], 1), 3)

    outputs = {
        'mapped_lab': out_dir / 'lab_names_with_candidate_cids.csv',
        'labeled': out_dir / 'lab_labeled_for_evaluation.csv',
        'ambiguous': out_dir / 'lab_names_ambiguous_cids.csv',
        'unmatched': out_dir / 'lab_names_unmatched.csv',
        'long_matches': out_dir / 'lab_name_column_matches.csv',
    }
    mapped_lab.to_csv(outputs['mapped_lab'], index=False)
    mapped_lab[mapped_lab['mapping_status'].eq('exact_unique_cid')].to_csv(outputs['labeled'], index=False)
    mapped_lab[mapped_lab['mapping_status'].eq('exact_ambiguous_cid')].to_csv(outputs['ambiguous'], index=False)
    mapped_lab[mapped_lab['mapping_status'].eq('unmatched')].to_csv(outputs['unmatched'], index=False)
    lab_matches_long.to_csv(outputs['long_matches'], index=False)

    summary = {
        'reference_summary': reference_summary,
        'lab_long_summary': lab_long_summary,
        'mapping_summary': mapping_summary,
        'lab_csv_path': str(LAB_CSV_PATH),
        'reference_csv_path': str(REFERENCE_CSV_PATH),
        'lab_name_columns': [METABOLITE_COL, REFMET_COL],
        'outputs': {k: str(v) for k, v in outputs.items()},
    }
    save_json(summary, summary_path)
    print(json.dumps(summary, indent=2))
    return summary


if RUN_DETERMINISTIC_ROW_CID_MAPPING:
    deterministic_mapping_summary = run_deterministic_row_cid_mapping()
else:
    deterministic_mapping_summary = {'skipped': True}

## Load Lab Pairs and RefMet Label Set


In [ ]:
def load_lab_pairs_and_targets() -> tuple[pd.DataFrame, pd.DataFrame]:
    lab = pd.read_csv(LAB_CSV_PATH, dtype='string', engine='python', on_bad_lines='skip')
    missing = [col for col in [METABOLITE_COL, REFMET_COL] if col not in lab.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}. Available columns: {list(lab.columns)}')

    lab = lab.reset_index(drop=True).copy()
    lab['source_row_id'] = np.arange(len(lab), dtype=np.int64)
    lab['metabolite_name'] = lab[METABOLITE_COL].astype('string').str.strip()
    lab['refmet_name'] = lab[REFMET_COL].astype('string').str.strip()
    lab['metabolite_normalized'] = normalize_series(lab['metabolite_name'])
    lab['refmet_normalized'] = normalize_series(lab['refmet_name'])
    keep = (lab['metabolite_normalized'].str.len() > 0) & (lab['refmet_normalized'].str.len() > 0)
    pairs = lab[keep].copy()
    pairs = pairs.drop_duplicates(['metabolite_normalized', 'refmet_normalized']).reset_index(drop=True)

    targets = (
        pairs.groupby('refmet_normalized', sort=True)
        .agg(refmet_name=('refmet_name', 'first'), n_query_pairs=('source_row_id', 'size'))
        .reset_index()
    )
    targets['target_id'] = np.arange(len(targets), dtype=np.int64)
    target_lookup = targets.set_index('refmet_normalized')['target_id'].to_dict()
    pairs['target_id'] = pairs['refmet_normalized'].map(target_lookup).astype(np.int64)
    pairs['query_id'] = np.arange(len(pairs), dtype=np.int64)

    rng = np.random.default_rng(SEED)
    order = rng.permutation(len(pairs))
    split = np.empty(len(pairs), dtype=object)
    n_train = int(round(TRAIN_FRAC * len(pairs)))
    n_val = int(round(VAL_FRAC * len(pairs)))
    split[order[:n_train]] = 'train'
    split[order[n_train:n_train+n_val]] = 'val'
    split[order[n_train+n_val:]] = 'test'
    pairs['split'] = split

    pairs.to_csv(RESULTS_ARTIFACTS_DIR / 'v4_query_pairs.csv', index=False)
    targets.to_csv(RESULTS_ARTIFACTS_DIR / 'v4_refmet_label_dictionary_initial.csv', index=False)

    summary = {
        'raw_lab_rows': int(len(lab)),
        'rows_with_both_names': int(keep.sum()),
        'deduplicated_query_pairs': int(len(pairs)),
        'unique_refmet_labels': int(len(targets)),
        'split_counts': pairs['split'].value_counts().to_dict(),
        'identical_metabolite_refmet_pairs': int(pairs['metabolite_normalized'].eq(pairs['refmet_normalized']).sum()),
    }
    save_json(summary, RESULTS_RUN_DIR / 'v4_data_summary.json')
    print(json.dumps(summary, indent=2))
    return pairs, targets


lab_pairs, refmet_targets = load_lab_pairs_and_targets()
display(lab_pairs.head(10))
display(refmet_targets.head(10))


## Build Targeted Reference Lookup Corpus for Identity Mapping

The final evaluation does not embed or search all 114M reference rows. For identity-aware scoring, the notebook converts the deterministic targeted lookup into a compact parquet table with one row per matched normalized lab surface.


In [ ]:
def prepare_unique_surface_corpus() -> dict:
    # Kept under the original function name so downstream V4 scoring code can be reused.
    lookup_csv = RESULTS_ARTIFACTS_DIR / 'targeted_reference_lookup.csv'
    if CORPUS_PARQUET_PATH.exists():
        pf = pq.ParquetFile(CORPUS_PARQUET_PATH)
        summary = {
            'loaded_from_cache': True,
            'source': 'targeted_reference_lookup',
            'unique_surface_rows': int(pf.metadata.num_rows),
            'corpus_parquet_path': str(CORPUS_PARQUET_PATH),
        }
        save_json(summary, RESULTS_RUN_DIR / 'v4_corpus_summary.json')
        print(json.dumps(summary, indent=2))
        return summary

    if not lookup_csv.exists():
        if RUN_DETERMINISTIC_ROW_CID_MAPPING:
            run_deterministic_row_cid_mapping()
        if not lookup_csv.exists():
            summary = {
                'loaded_from_cache': False,
                'corpus_available': False,
                'reason': 'targeted_reference_lookup_missing',
            }
            save_json(summary, RESULTS_RUN_DIR / 'v4_corpus_summary.json')
            print(json.dumps(summary, indent=2))
            return summary

    lookup = pd.read_csv(lookup_csv, dtype='string')
    if lookup.empty:
        summary = {
            'loaded_from_cache': False,
            'corpus_available': False,
            'reason': 'targeted_reference_lookup_empty',
        }
        save_json(summary, RESULTS_RUN_DIR / 'v4_corpus_summary.json')
        print(json.dumps(summary, indent=2))
        return summary

    corpus = lookup[['normalized_name', 'candidate_cids', 'n_candidate_cids']].drop_duplicates('normalized_name').copy()
    corpus = corpus.sort_values('normalized_name').reset_index(drop=True)
    corpus.insert(0, 'surface_id', np.arange(len(corpus), dtype=np.int64))
    corpus['display_name'] = corpus['normalized_name']
    corpus = corpus[['surface_id', 'normalized_name', 'display_name', 'candidate_cids', 'n_candidate_cids']]
    corpus.to_parquet(CORPUS_PARQUET_PATH, index=False)

    summary = {
        'loaded_from_cache': False,
        'source': str(lookup_csv),
        'unique_surface_rows': int(len(corpus)),
        'corpus_parquet_path': str(CORPUS_PARQUET_PATH),
    }
    save_json(summary, RESULTS_RUN_DIR / 'v4_corpus_summary.json')
    print(json.dumps(summary, indent=2))
    return summary


corpus_summary = prepare_unique_surface_corpus()

## Map RefMet Labels to PubChem CIDs


In [ ]:
def aggregate_cid_matches(matches: pd.DataFrame) -> pd.DataFrame:
    if matches.empty:
        return pd.DataFrame(columns=['target_id', 'refmet_mapping_method', 'mapped_refmet_names', 'refmet_candidate_cids', 'refmet_n_candidate_cids'])
    priority = {
        'exact': 0,
        'strip_instrument_suffix': 1,
        'punctuation_spaced': 2,
        'remove_quotes': 3,
        'remove_terminal_count': 4,
        'compact_key': 5,
    }
    matches = matches.copy()
    matches['priority'] = matches['method'].map(priority).fillna(99).astype(int)
    best = matches[matches['priority'].eq(matches.groupby('target_id')['priority'].transform('min'))].copy()
    rows = []
    for target_id, group in best.groupby('target_id', sort=False):
        cids = pipe_join(group['candidate_cids'])
        rows.append({
            'target_id': int(target_id),
            'refmet_mapping_method': '|'.join(pd.unique(group['method'].astype(str))),
            'mapped_refmet_names': '|'.join(pd.unique(group['mapped_name'].astype(str))[:20]),
            'refmet_candidate_cids': cids,
            'refmet_n_candidate_cids': len(pipe_split(cids)),
        })
    return pd.DataFrame(rows)


def map_refmet_targets_to_cids(targets: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    cache_path = RESULTS_ARTIFACTS_DIR / 'v4_refmet_label_cid_mapping.csv'
    if cache_path.exists():
        mapped = pd.read_csv(cache_path)
        out = targets.merge(mapped, on='target_id', how='left')
        out['refmet_mapped_to_cid'] = out['refmet_candidate_cids'].fillna('').astype(str).str.len() > 0
        summary = {
            'loaded_from_cache': True,
            'unique_refmet_labels': int(len(out)),
            'mapped_labels_total': int(out['refmet_mapped_to_cid'].sum()),
            'unmapped_labels_total': int((~out['refmet_mapped_to_cid']).sum()),
            'mapped_by_method': out.loc[out['refmet_mapped_to_cid'], 'refmet_mapping_method'].value_counts(dropna=False).to_dict(),
        }
        save_json(summary, RESULTS_RUN_DIR / 'v4_refmet_cid_mapping_summary.json')
        print(json.dumps(summary, indent=2))
        return out, summary

    if not CORPUS_PARQUET_PATH.exists():
        out = targets.copy()
        out['refmet_candidate_cids'] = ''
        out['refmet_n_candidate_cids'] = 0
        out['refmet_mapping_method'] = ''
        out['refmet_mapped_to_cid'] = False
        summary = {
            'loaded_from_cache': False,
            'unique_refmet_labels': int(len(out)),
            'mapped_labels_total': 0,
            'unmapped_labels_total': int(len(out)),
            'reason': 'unique_surface_corpus_missing',
        }
        save_json(summary, RESULTS_RUN_DIR / 'v4_refmet_cid_mapping_summary.json')
        return out, summary

    variant_rows = []
    for row in targets[['target_id', 'refmet_name']].itertuples(index=False):
        seen = set()
        for method, variant in make_refmet_variants(row.refmet_name):
            if variant and variant not in seen:
                variant_rows.append({'target_id': int(row.target_id), 'method': method, 'variant_name': variant})
                seen.add(variant)
    variants = pd.DataFrame(variant_rows).drop_duplicates(['target_id', 'method', 'variant_name'])

    con = duckdb.connect()
    con.register('variants', variants)
    variant_matches = con.execute(f'''
        SELECT
            v.target_id,
            v.method,
            u.normalized_name AS mapped_name,
            u.candidate_cids
        FROM variants v
        JOIN read_parquet('{q(CORPUS_PARQUET_PATH)}') u
            ON v.variant_name = u.normalized_name
    ''').df()

    all_matches = [variant_matches]
    mapped_ids = set(variant_matches['target_id'].astype(int).tolist()) if not variant_matches.empty else set()
    if RUN_COMPACT_REFMET_MAPPING:
        remaining = targets[~targets['target_id'].isin(mapped_ids)].copy()
        compact = remaining[['target_id', 'refmet_normalized']].copy()
        compact['compact_refmet_key'] = compact['refmet_normalized'].map(compact_key)
        compact = compact[compact['compact_refmet_key'].str.len() >= int(MIN_COMPACT_KEY_LEN)]
        compact = compact.drop_duplicates(['target_id', 'compact_refmet_key'])
        con.register('compact_refmets', compact[['target_id', 'compact_refmet_key']])
        compact_matches = con.execute(f'''
            WITH raw AS (
                SELECT
                    c.target_id,
                    'compact_key' AS method,
                    u.normalized_name AS mapped_name,
                    u.candidate_cids
                FROM read_parquet('{q(CORPUS_PARQUET_PATH)}') u
                JOIN compact_refmets c
                    ON regexp_replace(u.normalized_name, '[^a-z0-9]+', '', 'g') = c.compact_refmet_key
            ),
            counted AS (
                SELECT *, COUNT(*) OVER (PARTITION BY target_id) AS n_surface_matches
                FROM raw
            )
            SELECT target_id, method, mapped_name, candidate_cids
            FROM counted
            WHERE n_surface_matches <= {int(MAX_COMPACT_MATCHES_PER_REFMET)}
        ''').df()
        all_matches.append(compact_matches)
    con.close()

    matches = pd.concat(all_matches, ignore_index=True) if all_matches else pd.DataFrame()
    mapping = aggregate_cid_matches(matches)
    mapping.to_csv(cache_path, index=False)
    out = targets.merge(mapping, on='target_id', how='left')
    out['refmet_candidate_cids'] = out['refmet_candidate_cids'].fillna('')
    out['refmet_n_candidate_cids'] = out['refmet_n_candidate_cids'].fillna(0).astype(int)
    out['refmet_mapping_method'] = out['refmet_mapping_method'].fillna('')
    out['refmet_mapped_to_cid'] = out['refmet_candidate_cids'].astype(str).str.len() > 0
    summary = {
        'loaded_from_cache': False,
        'unique_refmet_labels': int(len(out)),
        'mapped_labels_total': int(out['refmet_mapped_to_cid'].sum()),
        'unmapped_labels_total': int((~out['refmet_mapped_to_cid']).sum()),
        'mapped_by_method': out.loc[out['refmet_mapped_to_cid'], 'refmet_mapping_method'].value_counts(dropna=False).to_dict(),
    }
    save_json(summary, RESULTS_RUN_DIR / 'v4_refmet_cid_mapping_summary.json')
    print(json.dumps(summary, indent=2))
    return out, summary


refmet_targets, refmet_cid_mapping_summary = map_refmet_targets_to_cids(refmet_targets)
refmet_targets.to_csv(RESULTS_ARTIFACTS_DIR / 'v4_refmet_label_dictionary.csv', index=False)

target_cids = refmet_targets['refmet_candidate_cids'].fillna('').astype(str).tolist()
target_cid_sets = [pipe_split(x) for x in target_cids]
target_norms = refmet_targets['refmet_normalized'].astype(str).tolist()
target_display = refmet_targets['refmet_name'].astype(str).tolist()
target_mapping_methods = refmet_targets['refmet_mapping_method'].fillna('').astype(str).tolist()


## Encode Queries and RefMet Labels


In [ ]:
import torch


def encode_texts_to_memmap(
    texts: list[str],
    out_path: Path,
    tokenizer,
    model,
    model_name: str,
    batch_size: int,
    device: str,
    run_key: str,
    force_reencode: bool = False,
) -> np.ndarray:
    n = len(texts)
    d = int(model.config.hidden_size)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    meta_path = out_path.with_suffix(out_path.suffix + '.meta.json')
    expected_meta = {
        'n': n,
        'd': d,
        'model_name': model_name,
        'pooling': POOLING,
        'max_length': int(MAX_LENGTH),
        'dtype': EMBED_DTYPE,
        'normalize': bool(L2_NORMALIZE),
        'run_key': run_key,
    }
    if out_path.exists() and meta_path.exists() and not force_reencode:
        old_meta = load_json_if_exists(meta_path)
        arr = np.load(out_path, mmap_mode='r')
        if old_meta == expected_meta and arr.shape == (n, d):
            print('Using cached embeddings:', out_path)
            return arr
    if out_path.exists():
        out_path.unlink()
    arr = np.lib.format.open_memmap(
        out_path,
        mode='w+',
        dtype=np.float16 if EMBED_DTYPE == 'float16' else np.float32,
        shape=(n, d),
    )
    autocast_enabled = device.startswith('cuda') and torch.cuda.is_available()
    t0 = time.perf_counter()
    with torch.inference_mode():
        for start in tqdm(range(0, n, int(batch_size)), desc=f'Encoding {out_path.name}'):
            end = min(start + int(batch_size), n)
            batch = [str(x) for x in texts[start:end]]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=int(MAX_LENGTH), return_tensors='pt').to(device)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=autocast_enabled):
                out = model(**enc)
                emb = pool_outputs(out.last_hidden_state, enc['attention_mask'], POOLING)
                if L2_NORMALIZE:
                    emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            arr[start:end] = emb.detach().cpu().numpy().astype(arr.dtype, copy=False)
            del batch, enc, out, emb
            if device.startswith('cuda'):
                torch.cuda.empty_cache()
    arr.flush()
    save_json(expected_meta, meta_path)
    save_json({'elapsed_sec': float(time.perf_counter() - t0)}, out_path.with_suffix(out_path.suffix + '.timing.json'))
    return np.load(out_path, mmap_mode='r')


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Embedding device:', device)
tokenizer, hf_model = load_hf_model(str(MODEL_DIR), device)

query_embeddings = encode_texts_to_memmap(
    lab_pairs['metabolite_normalized'].astype(str).tolist(),
    QUERY_EMBEDDINGS_PATH,
    tokenizer,
    hf_model,
    str(MODEL_DIR),
    QUERY_ENCODE_BATCH_SIZE,
    device,
    run_key=RUN_NAME + '_queries',
    force_reencode=REENCODE_QUERIES,
)
refmet_embeddings = encode_texts_to_memmap(
    refmet_targets['refmet_normalized'].astype(str).tolist(),
    REFMET_EMBEDDINGS_PATH,
    tokenizer,
    hf_model,
    str(MODEL_DIR),
    REFMET_ENCODE_BATCH_SIZE,
    device,
    run_key=RUN_NAME + '_refmet_labels',
    force_reencode=REENCODE_REFMET_LABELS,
)
del hf_model
del tokenizer
gc.collect()
if device.startswith('cuda'):
    torch.cuda.empty_cache()
print('Query embeddings:', query_embeddings.shape, query_embeddings.dtype)
print('RefMet label embeddings:', refmet_embeddings.shape, refmet_embeddings.dtype)


## Exact Closed-Set Search


In [ ]:
import faiss


def build_refmet_index(embeddings: np.ndarray, device: str):
    dim = embeddings.shape[1]
    cpu_index = faiss.IndexFlatIP(dim)
    resource = None
    index = cpu_index
    if device.startswith('cuda') and hasattr(faiss, 'StandardGpuResources'):
        resource = faiss.StandardGpuResources()
        opts = faiss.GpuClonerOptions()
        opts.useFloat16 = True
        index = faiss.index_cpu_to_gpu(resource, 0, cpu_index, opts)
    for start in tqdm(range(0, len(embeddings), 250000), desc='Adding RefMet labels to exact index'):
        end = min(start + 250000, len(embeddings))
        xb = np.ascontiguousarray(np.asarray(embeddings[start:end], dtype=np.float32))
        index.add(xb)
        del xb
    return index, resource


def search_index(index, query_embeddings: np.ndarray, k: int, batch_size: int) -> tuple[np.ndarray, np.ndarray]:
    n = len(query_embeddings)
    idx = np.lib.format.open_memmap(NEIGHBOR_INDICES_PATH, mode='w+', dtype=np.int64, shape=(n, int(k)))
    scores = np.lib.format.open_memmap(NEIGHBOR_SCORES_PATH, mode='w+', dtype=np.float32, shape=(n, int(k)))
    for start in tqdm(range(0, n, int(batch_size)), desc='Searching closed RefMet label set'):
        end = min(start + int(batch_size), n)
        xq = np.ascontiguousarray(np.asarray(query_embeddings[start:end], dtype=np.float32))
        d, i = index.search(xq, int(k))
        idx[start:end] = i
        scores[start:end] = d
        del xq, d, i
    idx.flush()
    scores.flush()
    return np.load(NEIGHBOR_INDICES_PATH, mmap_mode='r'), np.load(NEIGHBOR_SCORES_PATH, mmap_mode='r')


search_meta_path = NEIGHBOR_INDICES_PATH.with_suffix(NEIGHBOR_INDICES_PATH.suffix + '.meta.json')
expected_search_meta = {
    'n_queries': int(len(lab_pairs)),
    'n_targets': int(len(refmet_targets)),
    'k_retrieve': int(K_RETRIEVE),
    'query_shape': tuple(query_embeddings.shape),
    'target_shape': tuple(refmet_embeddings.shape),
    'run_name': RUN_NAME,
}
if (
    NEIGHBOR_INDICES_PATH.exists()
    and NEIGHBOR_SCORES_PATH.exists()
    and load_json_if_exists(search_meta_path) == expected_search_meta
    and not RERUN_SEARCH
):
    neighbor_indices = np.load(NEIGHBOR_INDICES_PATH, mmap_mode='r')
    neighbor_scores = np.load(NEIGHBOR_SCORES_PATH, mmap_mode='r')
    print('Using cached closed-set neighbors:', neighbor_indices.shape)
else:
    refmet_index, refmet_index_resource = build_refmet_index(refmet_embeddings, device)
    neighbor_indices, neighbor_scores = search_index(refmet_index, query_embeddings, K_RETRIEVE, SEARCH_BATCH_SIZE)
    save_json(expected_search_meta, search_meta_path)
print('Closed-set neighbors:', neighbor_indices.shape)


## Feature Builder and Identity-Aware Correctness


In [ ]:
target_profiles = [text_profile(x) for x in tqdm(target_norms, desc='Precomputing target text profiles')]
query_norms = lab_pairs['metabolite_normalized'].astype(str).tolist()
query_profiles = [text_profile(x) for x in tqdm(query_norms, desc='Precomputing query text profiles')]
gold_target_ids = lab_pairs['target_id'].to_numpy(dtype=np.int64)
gold_norms = lab_pairs['refmet_normalized'].astype(str).tolist()

identity_correct_cache: dict[tuple[int, int], tuple[bool, str]] = {}


def identity_correct_pair(gold_id: int, candidate_id: int) -> tuple[bool, str]:
    if gold_id == candidate_id:
        return True, 'exact_label'
    key = (int(gold_id), int(candidate_id))
    if key in identity_correct_cache:
        return identity_correct_cache[key]
    if target_cid_sets[gold_id] and target_cid_sets[candidate_id] and (target_cid_sets[gold_id] & target_cid_sets[candidate_id]):
        result = (True, 'same_pubchem_cid')
    elif strong_alias_match(target_norms[gold_id], target_norms[candidate_id]):
        result = (True, 'strong_alias')
    else:
        result = (False, '')
    identity_correct_cache[key] = result
    return result


def first_ranks_for_row(row_i: int, ordered_ids: np.ndarray) -> tuple[float, float, str]:
    gold = int(gold_target_ids[row_i])
    exact_rank = np.inf
    identity_rank = np.inf
    identity_reason = ''
    for rank0, target_id in enumerate(ordered_ids):
        target_id = int(target_id)
        if target_id == gold and not np.isfinite(exact_rank):
            exact_rank = rank0 + 1
        ok, reason = identity_correct_pair(gold, target_id)
        if ok and not np.isfinite(identity_rank):
            identity_rank = rank0 + 1
            identity_reason = reason
        if np.isfinite(exact_rank) and np.isfinite(identity_rank):
            break
    return exact_rank, identity_rank, identity_reason


def metrics_from_ranks(rank_arr: np.ndarray, prefix: str = '') -> dict:
    rank_arr = np.asarray(rank_arr, dtype=np.float64)
    finite = np.isfinite(rank_arr)
    n = len(rank_arr)
    out = {f'{prefix}MRR': float(np.sum(np.where(finite, 1.0 / rank_arr, 0.0)) / max(1, n))}
    for k in [1, 5, 10, 25, 50, 100, 256]:
        out[f'{prefix}HitRate@{k}'] = float(np.mean(rank_arr <= k)) if n else 0.0
    return out


FEATURE_NAMES = [
    'embedding_score_norm',
    'rank_prior',
    'compact_exact',
    'sequence_ratio',
    'char3_jaccard',
    'token_jaccard',
    'lipid_signature_match',
    'lipid_class_match',
    'stereo_mismatch',
]


def build_feature_block(row_indices: np.ndarray, pool_k: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    row_indices = np.asarray(row_indices, dtype=np.int64)
    cand_ids = np.asarray(neighbor_indices[row_indices, :pool_k], dtype=np.int64)
    emb_scores = np.asarray(neighbor_scores[row_indices, :pool_k], dtype=np.float32)
    n, k = cand_ids.shape
    features = np.zeros((n, k, len(FEATURE_NAMES)), dtype=np.float32)
    features[:, :, 0] = (emb_scores + 1.0) / 2.0
    features[:, :, 1] = 1.0 / (np.arange(k, dtype=np.float32)[None, :] + 1.0)
    for local_i, row_i in enumerate(tqdm(row_indices, desc='Building lexical feature block', leave=False)):
        qprof = query_profiles[int(row_i)]
        for j, target_id in enumerate(cand_ids[local_i]):
            if target_id < 0:
                continue
            feats = lexical_feature_dict(qprof, target_profiles[int(target_id)])
            features[local_i, j, 2] = feats['compact_exact']
            features[local_i, j, 3] = feats['sequence_ratio']
            features[local_i, j, 4] = feats['char3_jaccard']
            features[local_i, j, 5] = feats['token_jaccard']
            features[local_i, j, 6] = feats['lipid_signature_match']
            features[local_i, j, 7] = feats['lipid_class_match']
            features[local_i, j, 8] = feats['stereo_mismatch']
    return cand_ids, emb_scores, features


def score_features(features: np.ndarray, weights: dict) -> np.ndarray:
    w = np.array([weights.get(name, 0.0) for name in FEATURE_NAMES], dtype=np.float32)
    return np.tensordot(features, w, axes=([2], [0]))


def ranks_from_neighbor_ids(row_indices: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray, list[str]]:
    exact_ranks = np.full(len(row_indices), np.inf, dtype=np.float64)
    identity_ranks = np.full(len(row_indices), np.inf, dtype=np.float64)
    identity_reasons = [''] * len(row_indices)
    for out_i, row_i in enumerate(row_indices):
        ordered = np.asarray(neighbor_indices[int(row_i), :k], dtype=np.int64)
        exact_rank, identity_rank, reason = first_ranks_for_row(int(row_i), ordered)
        exact_ranks[out_i] = exact_rank
        identity_ranks[out_i] = identity_rank
        identity_reasons[out_i] = reason
    return exact_ranks, identity_ranks, identity_reasons


def ranks_from_scores(cand_ids: np.ndarray, scores: np.ndarray, row_indices: np.ndarray) -> tuple[np.ndarray, np.ndarray, list[str], np.ndarray]:
    exact_ranks = np.full(len(row_indices), np.inf, dtype=np.float64)
    identity_ranks = np.full(len(row_indices), np.inf, dtype=np.float64)
    identity_reasons = [''] * len(row_indices)
    order = np.argsort(-scores, axis=1)
    for local_i, row_i in enumerate(row_indices):
        ordered_ids = cand_ids[local_i, order[local_i]]
        exact_rank, identity_rank, reason = first_ranks_for_row(int(row_i), ordered_ids)
        exact_ranks[local_i] = exact_rank
        identity_ranks[local_i] = identity_rank
        identity_reasons[local_i] = reason
    return exact_ranks, identity_ranks, identity_reasons, order


## Baseline Metrics


In [ ]:
split_indices = {
    split: lab_pairs.index[lab_pairs['split'].eq(split)].to_numpy(dtype=np.int64)
    for split in ['train', 'val', 'test']
}

baseline_metrics = {}
for split, indices in split_indices.items():
    exact_ranks, identity_ranks, _ = ranks_from_neighbor_ids(indices, K_RETRIEVE)
    non_identical = ~lab_pairs.iloc[indices]['metabolite_normalized'].eq(lab_pairs.iloc[indices]['refmet_normalized']).to_numpy()
    baseline_metrics[split] = {
        **metrics_from_ranks(exact_ranks, prefix='embedding_exact_'),
        **metrics_from_ranks(identity_ranks, prefix='embedding_identity_'),
        **metrics_from_ranks(exact_ranks[non_identical], prefix='nonidentical_embedding_exact_'),
        **metrics_from_ranks(identity_ranks[non_identical], prefix='nonidentical_embedding_identity_'),
    }
print(json.dumps(baseline_metrics, indent=2))
save_json(baseline_metrics, RESULTS_RUN_DIR / 'v4_embedding_baseline_metrics.json')


## Tune Hybrid Reranker on Validation Split


In [ ]:
def evaluate_weight_vector(cand_ids: np.ndarray, features: np.ndarray, row_indices: np.ndarray, weights: dict) -> dict:
    scores = score_features(features, weights)
    exact_ranks, identity_ranks, _, _ = ranks_from_scores(cand_ids, scores, row_indices)
    return {
        **metrics_from_ranks(exact_ranks, prefix='exact_'),
        **metrics_from_ranks(identity_ranks, prefix='identity_'),
    }


def make_weight_trials(rng: np.random.Generator) -> list[dict]:
    fixed = [
        {'name': 'embedding_only', 'embedding_score_norm': 1.0},
        {
            'name': 'v3_best_seed',
            'embedding_score_norm': 0.5752420344210024,
            'rank_prior': 0.0005994043670919589,
            'compact_exact': 0.01288747144452344,
            'sequence_ratio': 0.15114859837435277,
            'char3_jaccard': 0.023118621330574914,
            'token_jaccard': 0.17794705391329962,
            'lipid_signature_match': 0.05850092214070367,
            'lipid_class_match': 0.000555894008451015,
            'stereo_mismatch': -0.0,
        },
        {
            'name': 'lexical_heavy',
            'embedding_score_norm': 0.45,
            'rank_prior': 0.02,
            'compact_exact': 0.10,
            'sequence_ratio': 0.13,
            'char3_jaccard': 0.08,
            'token_jaccard': 0.14,
            'lipid_signature_match': 0.06,
            'lipid_class_match': 0.02,
            'stereo_mismatch': -0.04,
        },
    ]
    names = [
        'embedding_score_norm',
        'rank_prior',
        'compact_exact',
        'sequence_ratio',
        'char3_jaccard',
        'token_jaccard',
        'lipid_signature_match',
        'lipid_class_match',
    ]
    trials = fixed.copy()
    alpha = np.array([8.0, 0.5, 0.8, 1.8, 0.8, 2.0, 0.7, 0.3])
    for i in range(int(TUNE_RANDOM_WEIGHT_TRIALS)):
        vals = rng.dirichlet(alpha)
        trial = {'name': f'random_{i:03d}'}
        trial.update({name: float(value) for name, value in zip(names, vals)})
        trial['stereo_mismatch'] = -float(rng.choice([0.0, 0.02, 0.04, 0.06]))
        trials.append(trial)
    return trials


rng = np.random.default_rng(SEED)
val_indices = split_indices['val']
if len(val_indices) > int(TUNE_VAL_SAMPLE_SIZE):
    tune_indices = np.sort(rng.choice(val_indices, size=int(TUNE_VAL_SAMPLE_SIZE), replace=False))
else:
    tune_indices = val_indices
print('Tuning on validation rows:', len(tune_indices))

tuning_path = RESULTS_RUN_DIR / 'v4_reranker_tuning_results.csv'
weights_path = RESULTS_RUN_DIR / 'v4_best_reranker_weights.json'
if tuning_path.exists() and weights_path.exists() and not RERUN_TUNING:
    tuning_results = pd.read_csv(tuning_path)
    best_weights = load_json_if_exists(weights_path)
    print('Loaded cached tuning results.')
else:
    tune_cand_ids, tune_emb_scores, tune_features = build_feature_block(tune_indices, RERANK_POOL_K)
    rows = []
    for trial in tqdm(make_weight_trials(rng), desc='Tuning reranker weights'):
        metrics = evaluate_weight_vector(tune_cand_ids, tune_features, tune_indices, trial)
        row = {'name': trial['name'], **{k: trial.get(k, 0.0) for k in FEATURE_NAMES}, **metrics}
        rows.append(row)
    tuning_results = pd.DataFrame(rows).sort_values(['exact_MRR', 'exact_HitRate@1', 'exact_HitRate@10'], ascending=False).reset_index(drop=True)
    best_weights = tuning_results.iloc[0][['name', *FEATURE_NAMES]].to_dict()
    tuning_results.to_csv(tuning_path, index=False)
    save_json(best_weights, weights_path)
display(tuning_results.head(15))
print('Best weights:', json.dumps(best_weights, indent=2))


## Evaluate Final Reranker on Validation and Test


In [ ]:
def evaluate_reranker_split(row_indices: np.ndarray, split_name: str, save_predictions: bool = False):
    row_indices = np.asarray(row_indices, dtype=np.int64)
    exact_rank_blocks = []
    identity_rank_blocks = []
    prediction_rows = []
    top_rows = []
    weight_dict = {name: float(best_weights.get(name, 0.0)) for name in FEATURE_NAMES}

    for start in tqdm(range(0, len(row_indices), int(RERANK_BATCH_SIZE)), desc=f'Evaluating reranker {split_name}'):
        batch_indices = row_indices[start:start+int(RERANK_BATCH_SIZE)]
        cand_ids, emb_scores, features = build_feature_block(batch_indices, RERANK_POOL_K)
        scores = score_features(features, weight_dict)
        exact_ranks, identity_ranks, identity_reasons, order = ranks_from_scores(cand_ids, scores, batch_indices)
        exact_rank_blocks.append(exact_ranks)
        identity_rank_blocks.append(identity_ranks)

        if save_predictions:
            for local_i, row_i in enumerate(batch_indices):
                row_i = int(row_i)
                qrow = lab_pairs.iloc[row_i]
                ordered = order[local_i]
                top_j = int(ordered[0])
                top_target_id = int(cand_ids[local_i, top_j])
                gold = int(qrow['target_id'])
                top_score = float(scores[local_i, top_j])
                top2_score = float(scores[local_i, ordered[1]]) if len(ordered) > 1 else float('nan')
                top_exact = top_target_id == gold
                top_identity, top_identity_reason = identity_correct_pair(gold, top_target_id)
                emb_exact_rank, emb_identity_rank, emb_identity_reason = ranks_from_neighbor_ids(np.array([row_i]), K_RETRIEVE)
                prediction_rows.append({
                    'query_id': int(qrow['query_id']),
                    'source_row_id': int(qrow['source_row_id']),
                    'split': split_name,
                    'metabolite_name': qrow['metabolite_name'],
                    'refmet_name': qrow['refmet_name'],
                    'metabolite_normalized': qrow['metabolite_normalized'],
                    'refmet_normalized': qrow['refmet_normalized'],
                    'gold_target_id': gold,
                    'gold_refmet_cids': target_cids[gold],
                    'gold_refmet_mapping_method': target_mapping_methods[gold],
                    'metabolite_equals_refmet': bool(qrow['metabolite_normalized'] == qrow['refmet_normalized']),
                    'embedding_exact_rank': None if not np.isfinite(emb_exact_rank[0]) else int(emb_exact_rank[0]),
                    'embedding_identity_rank': None if not np.isfinite(emb_identity_rank[0]) else int(emb_identity_rank[0]),
                    'embedding_identity_reason': emb_identity_reason[0],
                    'reranker_exact_rank': None if not np.isfinite(exact_ranks[local_i]) else int(exact_ranks[local_i]),
                    'reranker_identity_rank': None if not np.isfinite(identity_ranks[local_i]) else int(identity_ranks[local_i]),
                    'reranker_identity_reason': identity_reasons[local_i],
                    'top1_refmet_name': target_display[top_target_id],
                    'top1_refmet_normalized': target_norms[top_target_id],
                    'top1_target_id': top_target_id,
                    'top1_refmet_cids': target_cids[top_target_id],
                    'top1_refmet_mapping_method': target_mapping_methods[top_target_id],
                    'top1_exact_correct': bool(top_exact),
                    'top1_identity_correct': bool(top_identity),
                    'top1_identity_reason': top_identity_reason,
                    'top1_embedding_score': float(emb_scores[local_i, top_j]),
                    'top1_rerank_score': top_score,
                    'top2_rerank_score': top2_score,
                    'confidence_margin': float(top_score - top2_score) if np.isfinite(top2_score) else float('nan'),
                })
                for rank0, j in enumerate(ordered[:int(SAVE_TOP_K)]):
                    target_id = int(cand_ids[local_i, int(j)])
                    exact_ok = target_id == gold
                    identity_ok, identity_reason = identity_correct_pair(gold, target_id)
                    top_rows.append({
                        'query_id': int(qrow['query_id']),
                        'rank': int(rank0 + 1),
                        'target_id': target_id,
                        'candidate_refmet_name': target_display[target_id],
                        'candidate_refmet_normalized': target_norms[target_id],
                        'candidate_refmet_cids': target_cids[target_id],
                        'exact_correct': bool(exact_ok),
                        'identity_correct': bool(identity_ok),
                        'identity_reason': identity_reason,
                        'embedding_score': float(emb_scores[local_i, int(j)]),
                        'rerank_score': float(scores[local_i, int(j)]),
                    })

    exact_ranks = np.concatenate(exact_rank_blocks) if exact_rank_blocks else np.array([], dtype=np.float64)
    identity_ranks = np.concatenate(identity_rank_blocks) if identity_rank_blocks else np.array([], dtype=np.float64)
    non_identical = ~lab_pairs.iloc[row_indices]['metabolite_normalized'].eq(lab_pairs.iloc[row_indices]['refmet_normalized']).to_numpy()
    metrics = {
        **metrics_from_ranks(exact_ranks, prefix='rerank_exact_'),
        **metrics_from_ranks(identity_ranks, prefix='rerank_identity_'),
        **metrics_from_ranks(exact_ranks[non_identical], prefix='nonidentical_rerank_exact_'),
        **metrics_from_ranks(identity_ranks[non_identical], prefix='nonidentical_rerank_identity_'),
    }
    return metrics, pd.DataFrame(prediction_rows), pd.DataFrame(top_rows)


reranker_metrics = {}
val_metrics, _, _ = evaluate_reranker_split(split_indices['val'], 'val', save_predictions=False)
reranker_metrics['val'] = val_metrics
test_metrics, test_predictions, test_top_hits = evaluate_reranker_split(split_indices['test'], 'test', save_predictions=True)
reranker_metrics['test'] = test_metrics
print(json.dumps(reranker_metrics, indent=2))


## Confidence Curves and Error Samples


In [ ]:
def confidence_curve(predictions: pd.DataFrame, correctness_col: str, coverage_levels=(1.0, 0.9, 0.75, 0.5, 0.25, 0.1)) -> pd.DataFrame:
    df = predictions.copy()
    df['confidence_margin'] = df['confidence_margin'].fillna(-1e9)
    df = df.sort_values('confidence_margin', ascending=False).reset_index(drop=True)
    rows = []
    n = len(df)
    for coverage in coverage_levels:
        keep_n = max(1, int(round(n * coverage)))
        kept = df.iloc[:keep_n]
        rows.append({
            'correctness_definition': correctness_col,
            'coverage': float(coverage),
            'n_answered': int(keep_n),
            'top1_accuracy_when_answered': float(kept[correctness_col].mean()),
            'minimum_confidence_margin': float(kept['confidence_margin'].min()),
        })
    return pd.DataFrame(rows)


exact_confidence_curve = confidence_curve(test_predictions, 'top1_exact_correct')
identity_confidence_curve = confidence_curve(test_predictions, 'top1_identity_correct')
confidence_curves = pd.concat([exact_confidence_curve, identity_confidence_curve], ignore_index=True)
display(confidence_curves)


def take_sample(df: pd.DataFrame, label: str, n: int = 25) -> pd.DataFrame:
    out = df.head(n).copy()
    out.insert(0, 'sample_type', label)
    return out


samples = []
samples.append(take_sample(test_predictions[test_predictions['top1_exact_correct']].sort_values('confidence_margin', ascending=False), 'high_confidence_exact_correct'))
samples.append(take_sample(test_predictions[~test_predictions['top1_exact_correct'] & test_predictions['top1_identity_correct']].sort_values('confidence_margin', ascending=False), 'high_confidence_identity_only'))
samples.append(take_sample(test_predictions[~test_predictions['top1_identity_correct']].sort_values('confidence_margin', ascending=False), 'high_confidence_wrong_identity'))
samples.append(take_sample(test_predictions[~test_predictions['top1_exact_correct']].sort_values('confidence_margin', ascending=True), 'low_confidence_wrong_exact'))
samples.append(take_sample(test_predictions[~test_predictions['top1_exact_correct'] & test_predictions['reranker_exact_rank'].between(2, 10, inclusive='both')].sort_values('reranker_exact_rank'), 'exact_correct_in_top10_not_top1'))
samples.append(take_sample(test_predictions[test_predictions['reranker_exact_rank'].isna()], 'exact_not_found_top256'))
error_samples = pd.concat(samples, ignore_index=True)
display(error_samples.head(30))



def build_success_failure_examples(predictions: pd.DataFrame, per_category: int = 15) -> pd.DataFrame:
    columns = [
        'query_id', 'source_row_id', 'split',
        'metabolite_name', 'refmet_name', 'top1_refmet_name',
        'metabolite_normalized', 'refmet_normalized', 'top1_refmet_normalized',
        'top1_exact_correct', 'top1_identity_correct', 'top1_identity_reason',
        'embedding_exact_rank', 'embedding_identity_rank',
        'reranker_exact_rank', 'reranker_identity_rank',
        'gold_refmet_cids', 'top1_refmet_cids',
        'top1_embedding_score', 'top1_rerank_score', 'top2_rerank_score', 'confidence_margin',
    ]
    available_cols = [col for col in columns if col in predictions.columns]

    specs = [
        (
            'success_exact_top1_high_confidence',
            'The reranked top-1 label exactly matches the expected refmet_name.',
            predictions[predictions['top1_exact_correct']].sort_values('confidence_margin', ascending=False),
        ),
        (
            'success_identity_only_high_confidence',
            'Top-1 does not exactly match the label, but it is counted correct by same-CID or strong-alias identity scoring.',
            predictions[~predictions['top1_exact_correct'] & predictions['top1_identity_correct']].sort_values('confidence_margin', ascending=False),
        ),
        (
            'success_reranker_rescued_from_embedding',
            'Embedding retrieval had the correct label below rank 1, but the hybrid reranker moved it to top-1.',
            predictions[
                predictions['top1_exact_correct']
                & predictions['embedding_exact_rank'].notna()
                & predictions['embedding_exact_rank'].gt(1)
            ].sort_values(['embedding_exact_rank', 'confidence_margin'], ascending=[False, False]),
        ),
        (
            'failure_high_confidence_wrong_identity',
            'The system is confident but the top-1 prediction is not exact and not identity-correct; these are priority error-analysis cases.',
            predictions[~predictions['top1_identity_correct']].sort_values('confidence_margin', ascending=False),
        ),
        (
            'failure_correct_in_top10_not_top1',
            'The correct label is in the candidate list but not ranked first, so this is mainly a reranking/order error.',
            predictions[
                ~predictions['top1_exact_correct']
                & predictions['reranker_exact_rank'].between(2, 10, inclusive='both')
            ].sort_values(['reranker_exact_rank', 'confidence_margin'], ascending=[True, False]),
        ),
        (
            'failure_not_found_top256',
            'The correct exact label was not retrieved in the top-256 candidate pool, so reranking cannot fix it.',
            predictions[predictions['reranker_exact_rank'].isna()].sort_values('confidence_margin', ascending=False),
        ),
        (
            'failure_low_confidence_wrong_exact',
            'The exact label is wrong and the margin is low; these are candidates for abstention or manual review.',
            predictions[~predictions['top1_exact_correct']].sort_values('confidence_margin', ascending=True),
        ),
    ]

    rows = []
    for example_type, interpretation, df in specs:
        if df.empty:
            continue
        out = df.head(per_category).copy()
        out.insert(0, 'example_type', example_type)
        out.insert(1, 'example_interpretation', interpretation)
        rows.append(out[['example_type', 'example_interpretation', *available_cols]])
    if not rows:
        return pd.DataFrame(columns=['example_type', 'example_interpretation', *available_cols])
    return pd.concat(rows, ignore_index=True)


success_failure_examples = build_success_failure_examples(test_predictions, per_category=15)
display(success_failure_examples.groupby('example_type').size().reset_index(name='n_examples'))
display(success_failure_examples.head(30))


## Save Final Outputs


In [ ]:
comparison_rows = []
for split in ['train', 'val', 'test']:
    row = {'split': split}
    row.update(baseline_metrics[split])
    row.update(reranker_metrics.get(split, {}))
    comparison_rows.append(row)
comparison = pd.DataFrame(comparison_rows)

final_metrics = {
    'run_name': RUN_NAME,
    'model_key': 'sapbert_contrastive',
    'task': 'final_closed_set_lab_refmet_name_prediction_from_metabolite_name',
    'data_summary': load_json_if_exists(RESULTS_RUN_DIR / 'v4_data_summary.json'),
    'refmet_cid_mapping_summary': refmet_cid_mapping_summary,
    'embedding_baseline': baseline_metrics,
    'tuned_reranker': reranker_metrics,
    'best_reranker_weights': best_weights,
    'primary_test_result': {
        **baseline_metrics['test'],
        **reranker_metrics['test'],
    },
    'notes': [
        'Candidate labels are unique refmet_name values from the provided lab file.',
        'Exact scoring requires the predicted normalized refmet_name to match the gold normalized refmet_name.',
        'Identity-aware scoring also counts same PubChem CID when both labels map, and conservative strong-alias matches.',
        'Reranker weights are selected on validation data and reported on the held-out test split.',
        'Non-identical metrics remove rows where metabolite_name and refmet_name normalize identically.',
    ],
}
save_json(final_metrics, RESULTS_RUN_DIR / 'v4_final_metrics.json')
comparison.to_csv(RESULTS_RUN_DIR / 'v4_final_comparison.csv', index=False)
test_predictions.to_csv(RESULTS_RUN_DIR / 'v4_final_test_predictions.csv', index=False)
test_top_hits.to_csv(RESULTS_RUN_DIR / 'v4_final_test_top_hits.csv', index=False)
confidence_curves.to_csv(RESULTS_RUN_DIR / 'v4_confidence_curves.csv', index=False)
error_samples.to_csv(RESULTS_RUN_DIR / 'v4_error_analysis_samples.csv', index=False)
success_failure_examples.to_csv(RESULTS_RUN_DIR / 'final_success_failure_examples.csv', index=False)

artifact_manifest = pd.DataFrame([
    {'field': 'config', 'value': str(RESULTS_RUN_DIR / 'config.json')},
    {'field': 'metrics', 'value': str(RESULTS_RUN_DIR / 'v4_final_metrics.json')},
    {'field': 'comparison', 'value': str(RESULTS_RUN_DIR / 'v4_final_comparison.csv')},
    {'field': 'test_predictions', 'value': str(RESULTS_RUN_DIR / 'v4_final_test_predictions.csv')},
    {'field': 'test_top_hits', 'value': str(RESULTS_RUN_DIR / 'v4_final_test_top_hits.csv')},
    {'field': 'confidence_curves', 'value': str(RESULTS_RUN_DIR / 'v4_confidence_curves.csv')},
    {'field': 'error_samples', 'value': str(RESULTS_RUN_DIR / 'v4_error_analysis_samples.csv')},
    {'field': 'success_failure_examples', 'value': str(RESULTS_RUN_DIR / 'final_success_failure_examples.csv')},
    {'field': 'reranker_weights', 'value': str(RESULTS_RUN_DIR / 'v4_best_reranker_weights.json')},
    {'field': 'tuning_results', 'value': str(RESULTS_RUN_DIR / 'v4_reranker_tuning_results.csv')},
    {'field': 'refmet_dictionary', 'value': str(RESULTS_ARTIFACTS_DIR / 'v4_refmet_label_dictionary.csv')},
    {'field': 'deterministic_mapping_summary', 'value': str(RESULTS_RUN_DIR / 'deterministic_cid_mapping' / 'deterministic_mapping_summary.json')},
    {'field': 'targeted_reference_lookup', 'value': str(RESULTS_ARTIFACTS_DIR / 'targeted_reference_lookup.csv')},
    {'field': 'query_embeddings', 'value': str(QUERY_EMBEDDINGS_PATH)},
    {'field': 'refmet_label_embeddings', 'value': str(REFMET_EMBEDDINGS_PATH)},
    {'field': 'neighbor_indices', 'value': str(NEIGHBOR_INDICES_PATH)},
    {'field': 'neighbor_scores', 'value': str(NEIGHBOR_SCORES_PATH)},
])
artifact_manifest.to_csv(RESULTS_RUN_DIR / 'v4_artifact_manifest.csv', index=False)
display(comparison)
display(artifact_manifest)


## Final Plots


In [ ]:
import matplotlib.pyplot as plt

plot_dir = RESULTS_RUN_DIR / 'plots'
plot_dir.mkdir(parents=True, exist_ok=True)

comparison = pd.read_csv(RESULTS_RUN_DIR / 'v4_final_comparison.csv')
test_row = comparison[comparison['split'].eq('test')].iloc[0]

labels = ['Embedding exact', 'Rerank exact', 'Rerank identity-aware']
mrr_values = [
    test_row['embedding_exact_MRR'],
    test_row['rerank_exact_MRR'],
    test_row['rerank_identity_MRR'],
]
hit1_values = [
    test_row['embedding_exact_HitRate@1'],
    test_row['rerank_exact_HitRate@1'],
    test_row['rerank_identity_HitRate@1'],
]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, mrr_values)
ax.set_ylim(0, 1)
ax.set_ylabel('MRR')
ax.set_title('V4 final test MRR')
for i, value in enumerate(mrr_values):
    ax.text(i, value + 0.02, f'{value:.3f}', ha='center')
fig.tight_layout()
fig.savefig(plot_dir / 'v4_final_mrr.png', dpi=180, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, hit1_values)
ax.set_ylim(0, 1)
ax.set_ylabel('Hit@1')
ax.set_title('V4 final test top-1 accuracy')
for i, value in enumerate(hit1_values):
    ax.text(i, value + 0.02, f'{value:.1%}', ha='center')
fig.tight_layout()
fig.savefig(plot_dir / 'v4_final_hit1.png', dpi=180, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for name, group in confidence_curves.groupby('correctness_definition'):
    ax.plot(group['coverage'], group['top1_accuracy_when_answered'], marker='o', label=name)
ax.invert_xaxis()
ax.set_ylim(0, 1)
ax.set_xlabel('Coverage answered')
ax.set_ylabel('Top-1 accuracy')
ax.set_title('V4 confidence threshold curves')
ax.legend()
fig.tight_layout()
fig.savefig(plot_dir / 'v4_confidence_curves.png', dpi=180, bbox_inches='tight')
plt.show()

print('Saved plots to:', plot_dir)


## Canonical Figures, Source Table, and Research-Update Report

This final cell creates presentation-ready figures and a concise report inside the same run folder. These files are intended to be downloaded with the result folder and copied back into the repository.


In [ ]:
from textwrap import dedent

plot_dir = RESULTS_RUN_DIR / 'plots'
plot_dir.mkdir(parents=True, exist_ok=True)

metrics = load_json_if_exists(RESULTS_RUN_DIR / 'v4_final_metrics.json') or {}
data_summary = metrics.get('data_summary') or load_json_if_exists(RESULTS_RUN_DIR / 'v4_data_summary.json') or {}
refmet_map_summary = metrics.get('refmet_cid_mapping_summary') or load_json_if_exists(RESULTS_RUN_DIR / 'v4_refmet_cid_mapping_summary.json') or {}
det_summary = load_json_if_exists(RESULTS_RUN_DIR / 'deterministic_cid_mapping' / 'deterministic_mapping_summary.json') or {}
det_map = det_summary.get('mapping_summary', {})
primary = metrics.get('primary_test_result', {})
comparison = pd.read_csv(RESULTS_RUN_DIR / 'v4_final_comparison.csv')

# 1. Lab preparation counts.
prep_labels = ['Raw rows', 'Rows with both names', 'Deduplicated pairs', 'Unique RefMet labels']
prep_values = [
    data_summary.get('raw_lab_rows', np.nan),
    data_summary.get('rows_with_both_names', np.nan),
    data_summary.get('deduplicated_query_pairs', np.nan),
    data_summary.get('unique_refmet_labels', np.nan),
]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(prep_labels, prep_values, color=['#2563eb', '#0f766e', '#16a34a', '#7c3aed'])
ax.set_title('Lab-name cleaning and target construction')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=20)
for i, v in enumerate(prep_values):
    if pd.notna(v):
        ax.text(i, v, f'{int(v):,}', ha='center', va='bottom', fontsize=9)
fig.tight_layout()
fig.savefig(plot_dir / 'final_lab_preparation_counts.png', dpi=180, bbox_inches='tight')
plt.show()

# 2. Deterministic row-level CID mapping coverage.
map_labels = ['Unique CID', 'Ambiguous CID', 'Unmatched']
map_values = [det_map.get('unique_cid_rows', np.nan), det_map.get('ambiguous_cid_rows', np.nan), det_map.get('unmatched_rows', np.nan)]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(map_labels, map_values, color=['#16a34a', '#d97706', '#64748b'])
ax.set_title('Deterministic CID mapping status')
ax.set_ylabel('Lab rows')
for i, v in enumerate(map_values):
    if pd.notna(v):
        ax.text(i, v, f'{int(v):,}', ha='center', va='bottom', fontsize=9)
fig.tight_layout()
fig.savefig(plot_dir / 'final_deterministic_cid_mapping_coverage.png', dpi=180, bbox_inches='tight')
plt.show()

# 3. Test MRR comparison.
test_row = comparison[comparison['split'].eq('test')].iloc[0]
labels = ['Embedding exact', 'Rerank exact', 'Rerank identity-aware']
values = [test_row['embedding_exact_MRR'], test_row['rerank_exact_MRR'], test_row['rerank_identity_MRR']]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, values, color=['#2563eb', '#0f766e', '#16a34a'])
ax.set_ylim(0, 1)
ax.set_title('Final held-out test MRR')
ax.set_ylabel('MRR')
for i, v in enumerate(values):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)
fig.tight_layout()
fig.savefig(plot_dir / 'final_test_mrr_comparison.png', dpi=180, bbox_inches='tight')
plt.show()

# 4. Top-k performance.
topk = [1, 5, 10, 256]
series = {
    'Embedding exact': [test_row[f'embedding_exact_HitRate@{k}'] for k in topk],
    'Rerank exact': [test_row[f'rerank_exact_HitRate@{k}'] for k in topk],
    'Rerank identity-aware': [test_row[f'rerank_identity_HitRate@{k}'] for k in topk],
}
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(topk))
width = 0.25
for j, (name, vals) in enumerate(series.items()):
    ax.bar(x + (j - 1) * width, vals, width, label=name)
ax.set_xticks(x)
ax.set_xticklabels([f'Hit@{k}' for k in topk])
ax.set_ylim(0, 1)
ax.set_ylabel('Rate')
ax.set_title('Final held-out top-k performance')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(plot_dir / 'final_test_hitrate_topk.png', dpi=180, bbox_inches='tight')
plt.show()

# Number-source table for report discipline.
example_path = RESULTS_RUN_DIR / 'final_success_failure_examples.csv'

source_rows = [
    {'reported_value': 'raw_lab_rows', 'value': data_summary.get('raw_lab_rows'), 'source_file': str(RESULTS_RUN_DIR / 'v4_data_summary.json'), 'source_step': 'load_lab_pairs_and_targets'},
    {'reported_value': 'rows_with_both_names', 'value': data_summary.get('rows_with_both_names'), 'source_file': str(RESULTS_RUN_DIR / 'v4_data_summary.json'), 'source_step': 'load_lab_pairs_and_targets'},
    {'reported_value': 'deduplicated_query_pairs', 'value': data_summary.get('deduplicated_query_pairs'), 'source_file': str(RESULTS_RUN_DIR / 'v4_data_summary.json'), 'source_step': 'load_lab_pairs_and_targets'},
    {'reported_value': 'unique_refmet_labels', 'value': data_summary.get('unique_refmet_labels'), 'source_file': str(RESULTS_RUN_DIR / 'v4_data_summary.json'), 'source_step': 'load_lab_pairs_and_targets'},
    {'reported_value': 'deterministic_unique_cid_rows', 'value': det_map.get('unique_cid_rows'), 'source_file': str(RESULTS_RUN_DIR / 'deterministic_cid_mapping' / 'deterministic_mapping_summary.json'), 'source_step': 'run_deterministic_row_cid_mapping'},
    {'reported_value': 'deterministic_ambiguous_cid_rows', 'value': det_map.get('ambiguous_cid_rows'), 'source_file': str(RESULTS_RUN_DIR / 'deterministic_cid_mapping' / 'deterministic_mapping_summary.json'), 'source_step': 'run_deterministic_row_cid_mapping'},
    {'reported_value': 'deterministic_unmatched_rows', 'value': det_map.get('unmatched_rows'), 'source_file': str(RESULTS_RUN_DIR / 'deterministic_cid_mapping' / 'deterministic_mapping_summary.json'), 'source_step': 'run_deterministic_row_cid_mapping'},
    {'reported_value': 'refmet_labels_mapped_to_cid', 'value': refmet_map_summary.get('mapped_labels_total'), 'source_file': str(RESULTS_RUN_DIR / 'v4_refmet_cid_mapping_summary.json'), 'source_step': 'map_refmet_targets_to_cids'},
    {'reported_value': 'embedding_exact_test_MRR', 'value': primary.get('embedding_exact_MRR'), 'source_file': str(RESULTS_RUN_DIR / 'v4_final_metrics.json'), 'source_step': 'baseline closed-set evaluation'},
    {'reported_value': 'rerank_exact_test_MRR', 'value': primary.get('rerank_exact_MRR'), 'source_file': str(RESULTS_RUN_DIR / 'v4_final_metrics.json'), 'source_step': 'tuned reranker test evaluation'},
    {'reported_value': 'rerank_identity_test_MRR', 'value': primary.get('rerank_identity_MRR'), 'source_file': str(RESULTS_RUN_DIR / 'v4_final_metrics.json'), 'source_step': 'identity-aware reranker test evaluation'},
]
source_table = pd.DataFrame(source_rows)
source_table.to_csv(RESULTS_RUN_DIR / 'final_number_source_table.csv', index=False)

def fmt_count_value(value):
    if value is None:
        return '[VALUE TO VERIFY]'
    try:
        if pd.isna(value):
            return '[VALUE TO VERIFY]'
    except Exception:
        pass
    try:
        return f'{int(value):,}'
    except Exception:
        return str(value)

def fmt_float_value(value, digits=4):
    if value is None:
        return '[VALUE TO VERIFY]'
    try:
        return f'{float(value):.{digits}f}'
    except Exception:
        return '[VALUE TO VERIFY]'

def fmt_pct_value(value):
    if value is None:
        return '[VALUE TO VERIFY]'
    try:
        return f'{float(value):.2%}'
    except Exception:
        return '[VALUE TO VERIFY]'

report = f"""
# External Lab RefMet Final Reproducible Evaluation Report

## Summary

This run evaluates the corrected lab-name task: predict `refmet_name` from `metabolite_name` using the existing tuned SapBERT contrastive model plus a lightweight hybrid reranker. The notebook also performs deterministic row-level CID mapping against the PubMed/PubChem synonym reference and saves all outputs into one run directory.

## Data Preparation

- Raw lab rows: {fmt_count_value(data_summary.get('raw_lab_rows'))}
- Rows with both `metabolite_name` and `refmet_name`: {fmt_count_value(data_summary.get('rows_with_both_names'))}
- Deduplicated `metabolite_name -> refmet_name` pairs: {fmt_count_value(data_summary.get('deduplicated_query_pairs'))}
- Unique RefMet target labels: {fmt_count_value(data_summary.get('unique_refmet_labels'))}
- Split counts: {data_summary.get('split_counts', '[VALUE TO VERIFY]')}

Rows were removed from the closed-set evaluation if either side of the pair normalized to an empty string. Duplicate normalized query-target pairs were collapsed before splitting. Identical normalized `metabolite_name` and `refmet_name` pairs are retained in the full evaluation but are also reported separately through non-identical metrics.

## Deterministic CID Mapping

The deterministic bridge normalizes lab names from both configured columns and scans the large reference synonym CSV in chunks. A lab row is labeled `exact_unique_cid` when all matched normalized names collapse to one CID, `exact_ambiguous_cid` when matched names point to multiple CIDs, and `unmatched` when no normalized reference name is found.

- Unique-CID lab rows: {fmt_count_value(det_map.get('unique_cid_rows'))}
- Ambiguous-CID lab rows: {fmt_count_value(det_map.get('ambiguous_cid_rows'))}
- Unmatched lab rows: {fmt_count_value(det_map.get('unmatched_rows'))}
- Unique assigned CIDs: {fmt_count_value(det_map.get('unique_assigned_cids'))}

## Model Evaluation

Input: `metabolite_name`.
Expected output: the correct closed-set `refmet_name`. Identity-aware scoring additionally counts predictions that map to the same PubChem CID or a conservative strong alias when mapping evidence exists.

Held-out test results from this run:

- Embedding-only exact MRR: {fmt_float_value(primary.get('embedding_exact_MRR'))}
- Embedding-only exact Hit@1: {fmt_pct_value(primary.get('embedding_exact_HitRate@1'))}
- Reranker exact MRR: {fmt_float_value(primary.get('rerank_exact_MRR'))}
- Reranker exact Hit@1: {fmt_pct_value(primary.get('rerank_exact_HitRate@1'))}
- Reranker identity-aware MRR: {fmt_float_value(primary.get('rerank_identity_MRR'))}
- Reranker identity-aware Hit@1: {fmt_pct_value(primary.get('rerank_identity_HitRate@1'))}

## Success and Failure Examples

Representative examples are saved in `final_success_failure_examples.csv`. The table includes exact top-1 successes, identity-only successes, reranker-rescued cases, high-confidence wrong predictions, cases where the correct label is present in top-10 but not ranked first, and cases where the exact label is not found in the top-256 candidate pool.

## Interpretation

The embedding model functions as a candidate generator: it often places the correct target somewhere in the top-k set. The reranker improves the order of candidates by combining embedding similarity with lexical and chemical-name signals. It cannot recover a correct answer that was never retrieved into the candidate pool, so top-k recovery remains an upper bound on reranking performance.

## Publication-Readiness Checklist

1. Freeze this notebook, run configuration, split seed, and result folder as the canonical lab-name evaluation.
2. Repeat the run or bootstrap the held-out test predictions to add uncertainty intervals.
3. Manually audit high-confidence disagreements, especially lipid shorthand, stereochemistry, and abbreviation cases.
4. Decide whether exact RefMet label prediction or identity-aware same-compound scoring should be the primary endpoint.
5. If possible, validate the RefMet-to-PubChem mapping rules against an external/official mapping table.
""".strip()
(RESULTS_RUN_DIR / 'research_update_report.md').write_text(report, encoding='utf-8')

# Update manifest with final report/figures/source table.
manifest_path = RESULTS_RUN_DIR / 'v4_artifact_manifest.csv'
manifest = pd.read_csv(manifest_path) if manifest_path.exists() else pd.DataFrame(columns=['field', 'value'])
extra_manifest = pd.DataFrame([
    {'field': 'research_update_report', 'value': str(RESULTS_RUN_DIR / 'research_update_report.md')},
    {'field': 'number_source_table', 'value': str(RESULTS_RUN_DIR / 'final_number_source_table.csv')},
    {'field': 'success_failure_examples', 'value': str(example_path)},
    {'field': 'figure_lab_preparation_counts', 'value': str(plot_dir / 'final_lab_preparation_counts.png')},
    {'field': 'figure_cid_mapping_coverage', 'value': str(plot_dir / 'final_deterministic_cid_mapping_coverage.png')},
    {'field': 'figure_test_mrr_comparison', 'value': str(plot_dir / 'final_test_mrr_comparison.png')},
    {'field': 'figure_test_hitrate_topk', 'value': str(plot_dir / 'final_test_hitrate_topk.png')},
])
manifest = pd.concat([manifest, extra_manifest], ignore_index=True).drop_duplicates('field', keep='last')
manifest.to_csv(manifest_path, index=False)

print('Final report:', RESULTS_RUN_DIR / 'research_update_report.md')
print('Number source table:', RESULTS_RUN_DIR / 'final_number_source_table.csv')
print('Final artifact manifest:', manifest_path)
display(source_table)
display(manifest)